In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if tool_choice:
        params["tool_choice"] = tool_choice

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

In [4]:
messages: list[dict[str, str]] = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """,
)
response = chat(
    messages,
    tools=[web_search_schema],
    tool_choice={"type": "tool", "name": "web_search"},
)
print(response.usage.server_tool_use)
response

ServerToolUsage(web_fetch_requests=0, web_search_requests=2)


Message(id='msg_011CdBTAi8m7gnPafqTVuY9e', container=None, content=[ServerToolUseBlock(id='srvtoolu_01U5P5L15iyjJachYbWMy9ag', caller=None, input={'query': 'best exercises for gaining leg muscle'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EpECCioIERgCIiRiMmZlYjE1My00ODg1LTQ3OTEtOGI1OC04NDYxZDc1ZTJjYTYSDB7LQQ99seQ1JGAunxoM27eywsTpu/yPQBBwIjAn7ehIViZLtLIYymo4qDjwwpxVMXipjOi3ztKeyiCkqL+QSUeNr8WxozVgwZdWqWQqlAFtfTBTyqnV3+9UjeV1DwJ3+zhWOtFvqk4npqQO6HcFTHlDPmz1Lbch0WXHrwo2RtEizpL0a8+1X2GhabK1kxsbFxJPrY8BAA2nsFfPQZDvCVyI6DN7Jfvir505CU02r/Qjv4NMauqavJR0W9jAKPVxCt+SddIYrdON/JdiTx+mEU8ms8vBzs/HxkUwcu7P2l9rF35MGAM=', page_age=None, title='Maximizing Muscle Hypertrophy: A Systematic Review of ...', type='web_search_result', url='https://pmc.ncbi.nlm.nih.gov/articles/PMC6950543/'), WebSearchResultBlock(encrypted_content='EuwgCioIERgCIiRiMmZlYjE1My00ODg1LTQ3OTEtOGI1OC04NDYxZDc1ZTJjYTYSDG6